# Formal LSTM Cache Action Predictor

Goal: train a real LSTM model for cache/memory behavior, not a toy smoke test.

This notebook implements the design idea from the handwritten sketch:

```text
PC hash / instruction semantic / address / delta / hit-miss / time / cache state
        ↓
Embedding + LSTM memory
        ↓
multi-task heads
        ├── next useful delta class
        ├── future hit / miss tendency
        ├── cache bypass / low-priority insertion decision
        └── timing / reuse-distance bucket
        ↓
export action table for real ChampSim replay
```

Important framing: this is **not only an SPP filter**. The NN learns pattern + time + cache-state behavior and predicts a cache action. SPP/candidate fields may be used as extra input signals, but the model is not limited to keep/drop SPP.

The template direction follows public embedding-LSTM prefetcher code style, but this notebook changes the objective from only next-delta classification to multi-task cache-action learning: delta + future hit + bypass + timing.


## 0. Expected real data

Put real ChampSim-derived event tables under one of these paths:

```text
formal_NN_training/data/*.csv
formal_NN_training/data/*.parquet
formal_NN_training/data/generated/*.csv
formal_NN_training/data/generated/*.parquet
```

Minimum required columns:

```text
pc, addr
```

Recommended columns for the real experiment:

```text
trace, event_id, cycle, pc, addr, hit, is_store,
spp_delta, spp_conf,
mshr_occupancy, l2_occupancy, bandwidth_pressure,
semantic_class
```

No synthetic fallback is provided. If real data is missing, this notebook intentionally stops.


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

from pathlib import Path
import os, glob, math, json, time, warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

REPO_ROOT = Path.cwd()
if REPO_ROOT.name != 'formal_NN_training':
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'formal_NN_training').exists():
            REPO_ROOT = p / 'formal_NN_training'
            break

DATA_DIRS = [REPO_ROOT / 'data', REPO_ROOT / 'data' / 'generated']
ARTIFACT_DIR = REPO_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('REPO_ROOT =', REPO_ROOT)
print('DEVICE =', DEVICE)

@dataclass
class CFG:
    seq_len: int = 64
    stride: int = 8
    batch_size: int = 256
    epochs: int = 10
    lr: float = 2e-3
    weight_decay: float = 1e-5
    grad_clip: float = 1.0
    pc_buckets: int = 8192
    page_offset_buckets: int = 64
    delta_vocab_size: int = 256
    emb_dim: int = 32
    hidden_dim: int = 96
    num_layers: int = 2
    dropout: float = 0.15
    train_frac: float = 0.80
    val_frac: float = 0.10
    future_hit_horizon: int = 32
    bypass_reuse_threshold: int = 256
    timing_bins: tuple = (4, 8, 16, 32, 64, 128, 256)

cfg = CFG()
print(json.dumps(asdict(cfg), indent=2))


In [ ]:
# ============================================================
# 2. Load real event tables
# ============================================================

def read_table(path: Path) -> pd.DataFrame:
    if path.suffix == '.parquet':
        df = pd.read_parquet(path)
    elif path.suffix == '.csv':
        df = pd.read_csv(path)
    else:
        raise ValueError(path)
    df['source_file'] = str(path)
    return df

paths = []
for d in DATA_DIRS:
    paths += sorted(d.glob('*.csv'))
    paths += sorted(d.glob('*.parquet'))

if not paths:
    raise FileNotFoundError('No real ChampSim event table found. Put CSV/Parquet under formal_NN_training/data or data/generated.')

dfs = [read_table(p) for p in paths]
df = pd.concat(dfs, ignore_index=True)
print('Loaded rows:', len(df))
print('Files:', [str(p) for p in paths])
print('Columns:', list(df.columns))

required = {'pc', 'addr'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')


In [ ]:
# ============================================================
# 3. Feature engineering
# ============================================================

def parse_int_maybe_hex(x):
    if pd.isna(x):
        return 0
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float):
        return int(x)
    s = str(x).strip()
    try:
        return int(s, 16) if s.startswith(('0x', '0X')) else int(float(s))
    except Exception:
        return abs(hash(s))

df = df.copy()
df['pc_int'] = df['pc'].map(parse_int_maybe_hex).astype('int64')
df['addr_int'] = df['addr'].map(parse_int_maybe_hex).astype('int64')
df['line'] = (df['addr_int'] // 64).astype('int64')
df['page_offset'] = (df['line'] % cfg.page_offset_buckets).astype('int64')
df['pc_bucket'] = (df['pc_int'] % cfg.pc_buckets).astype('int64')

if 'trace' not in df.columns:
    df['trace'] = df['source_file']
if 'cycle' in df.columns:
    df = df.sort_values(['trace', 'cycle']).reset_index(drop=True)
elif 'event_id' in df.columns:
    df = df.sort_values(['trace', 'event_id']).reset_index(drop=True)
else:
    df = df.sort_values(['trace']).reset_index(drop=True)

df['prev_line'] = df.groupby('trace')['line'].shift(1)
df['delta'] = (df['line'] - df['prev_line']).fillna(0).clip(-4096, 4096).astype('int64')

# Build delta vocabulary from most common observed deltas.
top_deltas = df['delta'].value_counts().head(cfg.delta_vocab_size - 1).index.tolist()
delta_to_idx = {int(d): i for i, d in enumerate(top_deltas)}
UNK_DELTA = len(delta_to_idx)
df['delta_idx'] = df['delta'].map(lambda x: delta_to_idx.get(int(x), UNK_DELTA)).astype('int64')

# Optional dynamic state fields. Missing fields become zero, so real logs can grow incrementally.
for col in ['hit', 'is_store', 'spp_delta', 'spp_conf', 'mshr_occupancy', 'l2_occupancy', 'bandwidth_pressure']:
    if col not in df.columns:
        df[col] = 0

if 'semantic_class' not in df.columns:
    df['semantic_class'] = 'unknown'
semantic_vocab = {v: i for i, v in enumerate(sorted(df['semantic_class'].astype(str).unique()))}
df['semantic_idx'] = df['semantic_class'].astype(str).map(semantic_vocab).astype('int64')

num_cols = ['hit', 'is_store', 'spp_conf', 'mshr_occupancy', 'l2_occupancy', 'bandwidth_pressure']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype('float32')

print('Top-10 deltas:', top_deltas[:10])
print('Delta vocab size including UNK:', UNK_DELTA + 1)
print('Semantic classes:', semantic_vocab)
df.head()


In [ ]:
# ============================================================
# 4. Labels: delta, future hit, bypass, timing
# ============================================================

# Next useful delta: next observed line delta.
df['label_delta_idx'] = df.groupby('trace')['delta_idx'].shift(-1).fillna(UNK_DELTA).astype('int64')

# Future hit tendency: whether any hit appears in the next H events of this trace.
def future_any_hit(s, horizon):
    arr = s.to_numpy(dtype=np.float32)
    out = np.zeros(len(arr), dtype=np.float32)
    # vector-friendly enough for offline training table construction
    for k in range(1, horizon + 1):
        shifted = np.zeros_like(arr)
        shifted[:-k] = arr[k:]
        out = np.maximum(out, shifted)
    return pd.Series(out, index=s.index)

df['label_future_hit'] = df.groupby('trace')['hit'].apply(lambda s: future_any_hit(s, cfg.future_hit_horizon)).reset_index(level=0, drop=True).astype('float32')

# Reuse distance in events for the same cache line. If far/no reuse, bypass is desirable.
df['next_same_line_pos'] = np.nan
for trace, g in df.groupby('trace', sort=False):
    next_pos = {}
    vals = np.full(len(g), np.nan, dtype=np.float32)
    lines = g['line'].to_numpy()
    idxs = g.index.to_numpy()
    for local_i in range(len(g) - 1, -1, -1):
        line = int(lines[local_i])
        if line in next_pos:
            vals[local_i] = next_pos[line] - local_i
        next_pos[line] = local_i
    df.loc[idxs, 'reuse_distance'] = vals

df['reuse_distance'] = df['reuse_distance'].fillna(1e9).astype('float32')
df['label_bypass'] = (df['reuse_distance'] > cfg.bypass_reuse_threshold).astype('float32')

bins = np.array(cfg.timing_bins, dtype=np.float32)
df['label_timing_bucket'] = np.digitize(df['reuse_distance'].to_numpy(), bins, right=False).astype('int64')
NUM_TIMING_BUCKETS = len(cfg.timing_bins) + 1

print(df[['delta', 'label_delta_idx', 'hit', 'label_future_hit', 'reuse_distance', 'label_bypass', 'label_timing_bucket']].head(10))
print('Bypass rate:', df['label_bypass'].mean())


In [ ]:
# ============================================================
# 5. Sequence dataset
# ============================================================

cat_cols = ['pc_bucket', 'page_offset', 'delta_idx', 'semantic_idx']
num_cols = ['hit', 'is_store', 'spp_conf', 'mshr_occupancy', 'l2_occupancy', 'bandwidth_pressure']
label_cols = ['label_delta_idx', 'label_future_hit', 'label_bypass', 'label_timing_bucket']

class CacheSeqDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, starts: List[int]):
        self.frame = frame.reset_index(drop=True)
        self.starts = starts
        self.cat = self.frame[cat_cols].to_numpy(dtype=np.int64)
        self.num = self.frame[num_cols].to_numpy(dtype=np.float32)
        self.y_delta = self.frame['label_delta_idx'].to_numpy(dtype=np.int64)
        self.y_hit = self.frame['label_future_hit'].to_numpy(dtype=np.float32)
        self.y_bypass = self.frame['label_bypass'].to_numpy(dtype=np.float32)
        self.y_time = self.frame['label_timing_bucket'].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = self.starts[idx]
        e = s + cfg.seq_len
        target = e - 1
        return {
            'cat': torch.from_numpy(self.cat[s:e]),
            'num': torch.from_numpy(self.num[s:e]),
            'y_delta': torch.tensor(self.y_delta[target], dtype=torch.long),
            'y_hit': torch.tensor(self.y_hit[target], dtype=torch.float32),
            'y_bypass': torch.tensor(self.y_bypass[target], dtype=torch.float32),
            'y_time': torch.tensor(self.y_time[target], dtype=torch.long),
        }

def make_starts(frame: pd.DataFrame):
    starts = []
    base = 0
    for _, g in frame.groupby('trace', sort=False):
        n = len(g)
        for s in range(0, max(0, n - cfg.seq_len), cfg.stride):
            starts.append(base + s)
        base += n
    return starts

n = len(df)
n_train = int(n * cfg.train_frac)
n_val = int(n * (cfg.train_frac + cfg.val_frac))
train_df = df.iloc[:n_train].copy()
val_df = df.iloc[n_train:n_val].copy()
test_df = df.iloc[n_val:].copy()

train_ds = CacheSeqDataset(train_df, make_starts(train_df))
val_ds = CacheSeqDataset(val_df, make_starts(val_df))
test_ds = CacheSeqDataset(test_df, make_starts(test_df))

if len(train_ds) == 0 or len(val_ds) == 0:
    raise ValueError('Not enough rows for seq_len. Reduce cfg.seq_len or provide more real trace events.')

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

print('sequences:', len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# ============================================================
# 6. LSTM model: embedding memory + cache-action heads
# ============================================================

class LSTMCacheActionPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.pc_emb = nn.Embedding(cfg.pc_buckets, cfg.emb_dim)
        self.off_emb = nn.Embedding(cfg.page_offset_buckets, cfg.emb_dim // 2)
        self.delta_emb = nn.Embedding(cfg.delta_vocab_size, cfg.emb_dim)
        self.sem_emb = nn.Embedding(max(1, len(semantic_vocab)), cfg.emb_dim // 2)
        in_dim = cfg.emb_dim + cfg.emb_dim // 2 + cfg.emb_dim + cfg.emb_dim // 2 + len(num_cols)
        self.lstm = nn.LSTM(
            input_size=in_dim, hidden_size=cfg.hidden_dim, num_layers=cfg.num_layers,
            batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0.0
        )
        self.norm = nn.LayerNorm(cfg.hidden_dim)
        self.trunk = nn.Sequential(
            nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.ReLU(), nn.Dropout(cfg.dropout)
        )
        self.delta_head = nn.Linear(cfg.hidden_dim, cfg.delta_vocab_size)
        self.hit_head = nn.Linear(cfg.hidden_dim, 1)
        self.bypass_head = nn.Linear(cfg.hidden_dim, 1)
        self.time_head = nn.Linear(cfg.hidden_dim, NUM_TIMING_BUCKETS)

    def forward(self, cat, num):
        pc = self.pc_emb(cat[:, :, 0].clamp(0, cfg.pc_buckets - 1))
        off = self.off_emb(cat[:, :, 1].clamp(0, cfg.page_offset_buckets - 1))
        de = self.delta_emb(cat[:, :, 2].clamp(0, cfg.delta_vocab_size - 1))
        se = self.sem_emb(cat[:, :, 3].clamp(0, max(0, len(semantic_vocab) - 1)))
        x = torch.cat([pc, off, de, se, num], dim=-1)
        out, _ = self.lstm(x)
        h = self.trunk(self.norm(out[:, -1, :]))
        return {
            'delta': self.delta_head(h),
            'hit': self.hit_head(h).squeeze(-1),
            'bypass': self.bypass_head(h).squeeze(-1),
            'time': self.time_head(h),
        }

model = LSTMCacheActionPredictor().to(DEVICE)
print(model)
print('parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
# ============================================================
# 7. Training and validation
# ============================================================

ce = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, cfg.epochs))

def move(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

def compute_loss(pred, b):
    loss_delta = ce(pred['delta'], b['y_delta'])
    loss_hit = bce(pred['hit'], b['y_hit'])
    loss_bypass = bce(pred['bypass'], b['y_bypass'])
    loss_time = ce(pred['time'], b['y_time'])
    loss = loss_delta + 0.5 * loss_hit + 0.5 * loss_bypass + 0.3 * loss_time
    return loss, {'delta': loss_delta.item(), 'hit': loss_hit.item(), 'bypass': loss_bypass.item(), 'time': loss_time.item()}

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total = 0
    d_ok = h_ok = b_ok = t_ok = 0
    loss_sum = 0.0
    for batch in loader:
        b = move(batch)
        pred = model(b['cat'], b['num'])
        loss, _ = compute_loss(pred, b)
        bs = b['y_delta'].numel()
        total += bs
        loss_sum += loss.item() * bs
        d_ok += (pred['delta'].argmax(-1) == b['y_delta']).sum().item()
        h_ok += ((torch.sigmoid(pred['hit']) > 0.5).float() == b['y_hit']).sum().item()
        b_ok += ((torch.sigmoid(pred['bypass']) > 0.5).float() == b['y_bypass']).sum().item()
        t_ok += (pred['time'].argmax(-1) == b['y_time']).sum().item()
    return {
        'loss': loss_sum / max(total, 1),
        'delta_acc': d_ok / max(total, 1),
        'future_hit_acc': h_ok / max(total, 1),
        'bypass_acc': b_ok / max(total, 1),
        'timing_acc': t_ok / max(total, 1),
    }

history = []
best_val = -1.0
best_path = ARTIFACT_DIR / 'lstm_cache_action_predictor.pt'

for epoch in range(1, cfg.epochs + 1):
    model.train()
    t0 = time.time()
    running = 0.0
    seen = 0
    for batch in train_loader:
        b = move(batch)
        pred = model(b['cat'], b['num'])
        loss, parts = compute_loss(pred, b)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()
        bs = b['y_delta'].numel()
        running += loss.item() * bs
        seen += bs
    sched.step()
    val = evaluate(val_loader)
    score = 0.45 * val['delta_acc'] + 0.20 * val['future_hit_acc'] + 0.20 * val['bypass_acc'] + 0.15 * val['timing_acc']
    row = {'epoch': epoch, 'train_loss': running / max(seen, 1), **val, 'score': score, 'sec': time.time() - t0}
    history.append(row)
    print(row)
    if score > best_val:
        best_val = score
        torch.save({'model': model.state_dict(), 'cfg': asdict(cfg), 'delta_to_idx': delta_to_idx, 'semantic_vocab': semantic_vocab}, best_path)

hist_df = pd.DataFrame(history)
hist_df.to_csv(ARTIFACT_DIR / 'lstm_training_history.csv', index=False)
print('best checkpoint:', best_path)
hist_df.tail()


In [ ]:
# ============================================================
# 8. Test metrics and action-table export
# ============================================================

ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print('test:', evaluate(test_loader))

@torch.no_grad()
def export_actions(loader, out_csv):
    model.eval()
    rows = []
    for batch in loader:
        b = move(batch)
        pred = model(b['cat'], b['num'])
        delta_prob = torch.softmax(pred['delta'], dim=-1)
        time_prob = torch.softmax(pred['time'], dim=-1)
        rows.append(pd.DataFrame({
            'pred_delta_idx': delta_prob.argmax(-1).cpu().numpy(),
            'pred_delta_conf': delta_prob.max(-1).values.cpu().numpy(),
            'pred_future_hit_prob': torch.sigmoid(pred['hit']).cpu().numpy(),
            'pred_bypass_prob': torch.sigmoid(pred['bypass']).cpu().numpy(),
            'pred_timing_bucket': time_prob.argmax(-1).cpu().numpy(),
            'true_delta_idx': b['y_delta'].cpu().numpy(),
            'true_future_hit': b['y_hit'].cpu().numpy(),
            'true_bypass': b['y_bypass'].cpu().numpy(),
            'true_timing_bucket': b['y_time'].cpu().numpy(),
        }))
    out = pd.concat(rows, ignore_index=True)
    out.to_csv(out_csv, index=False)
    return out

action_table = export_actions(test_loader, ARTIFACT_DIR / 'lstm_cache_action_table.csv')
print('wrote', ARTIFACT_DIR / 'lstm_cache_action_table.csv')
action_table.head()


## 9. How this connects to the next ChampSim stage

After real training, the output table should be connected to a ChampSim replay script. The minimum next step is:

```text
1. generate real event CSV/Parquet from ChampSim
2. train this notebook
3. export lstm_cache_action_table.csv
4. write a .sh replay that maps each event/candidate to:
   - predicted delta
   - predicted future-hit probability
   - bypass probability
   - timing bucket
5. compare against no-prefetch / SPP / previous GRU / future Transformer
```

The real paper-level claim should not be based only on classification accuracy. Final claim must use IPC, MPKI, accuracy, coverage, timeliness, pollution, and bandwidth/MSHR pressure.
